In [ ]:
import duckdb
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

PROJECT_ROOT = Path.cwd()
RAW_CSV = PROJECT_ROOT / "data" / "raw" / "online_retail.csv"
DB_PATH = PROJECT_ROOT / "data" / "retail.duckdb"

print("Rådata finns:", RAW_CSV.exists())

In [ ]:
df_raw = pd.read_csv(
    RAW_CSV,
    encoding="ISO-8859-1",
    dtype={"InvoiceNo": str, "StockCode": str, "CustomerID": "Int64"},
    parse_dates=["InvoiceDate"],
    date_format="%m/%d/%Y %H:%M",
)

print(f"Rader: {len(df_raw):,}")
print(f"Kolumner: {list(df_raw.columns)}")
df_raw.head()

In [ ]:
report = {
    "Totalt antal rader": len(df_raw),
    "Saknad CustomerID": df_raw["CustomerID"].isna().sum(),
    "Saknad Description": df_raw["Description"].isna().sum(),
    "Exakta dubbletter": df_raw.duplicated().sum(),
    "Negativ Quantity": (df_raw["Quantity"] < 0).sum(),
    "Noll eller negativt UnitPrice": (df_raw["UnitPrice"] <= 0).sum(),
    "Makulerade ordrar (C-prefix)": df_raw["InvoiceNo"].str.startswith("C").sum(),
    "Unika kunder": df_raw["CustomerID"].nunique(),
    "Unika produkter": df_raw["StockCode"].nunique(),
    "Unika länder": df_raw["Country"].nunique(),
}

for k, v in report.items():
    print(f"{k}: {v}")

In [ ]:
df = df_raw.copy()

# ta bort exakta dubbletter
before = len(df)
df = df.drop_duplicates()
print(f"Dubbletter borttagna: {before - len(df):,}")

# rader utan produktbeskrivning är oanvändbara i analysen
before = len(df)
df = df[df["Description"].notna()]
print(f"Rader utan Description borttagna: {before - len(df):,}")

df["Description"] = df["Description"].str.strip().str.upper()
df["StockCode"] = df["StockCode"].str.strip().str.upper()
df["Country"] = df["Country"].str.strip()

# makuleringar är värdefull info, så de flaggas istället för att tas bort
df["is_cancellation"] = df["InvoiceNo"].str.startswith("C")

# noll-/negativt pris som inte är en makulering är administrativt skräp
before = len(df)
df = df[~((df["UnitPrice"] <= 0) & (~df["is_cancellation"]))]
print(f"Icke-transaktioner borttagna: {before - len(df):,}")

df["revenue"] = df["Quantity"] * df["UnitPrice"]

df = df.rename(columns={
    "InvoiceNo": "invoice_no",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "UnitPrice": "unit_price",
    "CustomerID": "customer_id",
    "Country": "country",
})

print(f"Slutligt antal rader: {len(df):,}")
df.head()

In [ ]:
print("Datumintervall:", df["invoice_date"].min(), "→", df["invoice_date"].max())
print("Total intäkt (GBP):", f"{df['revenue'].sum():,.0f}")
print("Andel rader utan kund-ID:", f"{df['customer_id'].isna().mean():.1%}")
df.dtypes

In [ ]:
if DB_PATH.exists():
    DB_PATH.unlink()

con = duckdb.connect(str(DB_PATH))
con.register("staging_raw", df)

con.execute("""
    CREATE OR REPLACE TABLE staging AS
    SELECT * FROM staging_raw
""")

print(con.execute("SELECT COUNT(*) FROM staging").fetchone()[0], "rader i staging")

In [ ]:
con.execute("""
-- dim_product: väljer den vanligaste beskrivningen per produktkod
CREATE OR REPLACE TABLE dim_product AS
WITH ranked AS (
    SELECT
        stock_code,
        description,
        COUNT(*) AS n,
        ROW_NUMBER() OVER (
            PARTITION BY stock_code
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM staging
    GROUP BY stock_code, description
)
SELECT
    ROW_NUMBER() OVER (ORDER BY stock_code) AS product_key,
    stock_code,
    description
FROM ranked
WHERE rn = 1;
""")

con.execute("""
-- dim_customer: samma princip, väljer det land som förekommer oftast per kund
CREATE OR REPLACE TABLE dim_customer AS
WITH ranked AS (
    SELECT
        customer_id,
        country,
        COUNT(*) AS n,
        ROW_NUMBER() OVER (
            PARTITION BY customer_id
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM staging
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id, country
)
SELECT
    ROW_NUMBER() OVER (ORDER BY customer_id) AS customer_key,
    customer_id,
    country
FROM ranked
WHERE rn = 1;
""")

con.execute("""
-- dim_date
CREATE OR REPLACE TABLE dim_date AS
WITH all_dates AS (
    SELECT DISTINCT CAST(invoice_date AS DATE) AS full_date
    FROM staging
)
SELECT
    CAST(STRFTIME(full_date, '%Y%m%d') AS INTEGER) AS date_key,
    full_date,
    EXTRACT(year    FROM full_date) AS year,
    EXTRACT(quarter FROM full_date) AS quarter,
    EXTRACT(month   FROM full_date) AS month,
    STRFTIME(full_date, '%B')       AS month_name,
    EXTRACT(day     FROM full_date) AS day_of_month,
    EXTRACT(dow     FROM full_date) AS day_of_week,
    STRFTIME(full_date, '%A')       AS day_name,
    CASE WHEN EXTRACT(dow FROM full_date) IN (0, 6)
         THEN TRUE ELSE FALSE END   AS is_weekend,
    DATE_TRUNC('month', full_date)  AS month_start
FROM all_dates
ORDER BY full_date;
""")

for t in ["dim_product", "dim_customer", "dim_date"]:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"{t}: {n:,} rader")

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE fact_sales AS
SELECT
    ROW_NUMBER() OVER (ORDER BY s.invoice_date, s.invoice_no) AS sales_key,
    s.invoice_no,
    p.product_key,
    c.customer_key,
    CAST(STRFTIME(CAST(s.invoice_date AS DATE), '%Y%m%d') AS INTEGER) AS date_key,
    s.invoice_date,
    s.customer_id,
    s.country,
    s.quantity,
    s.unit_price,
    s.revenue,
    s.is_cancellation
FROM staging s
LEFT JOIN dim_product  p ON s.stock_code  = p.stock_code
LEFT JOIN dim_customer c ON s.customer_id = c.customer_id;
""")

n = con.execute("SELECT COUNT(*) FROM fact_sales").fetchone()[0]
print(f"fact_sales: {n:,} rader")

orphans = con.execute("""
    SELECT COUNT(*) FROM fact_sales WHERE product_key IS NULL
""").fetchone()[0]
print(f"Rader utan produktmatchning: {orphans:,}")

In [ ]:
con.execute("DROP TABLE IF EXISTS staging")

print(con.execute("SHOW TABLES").fetchdf())
con.execute("SELECT * FROM fact_sales LIMIT 5").fetchdf()

In [ ]:
con.close()
print("Databas byggd:", DB_PATH)